# Object Tracking with OpenCV — Google Colab Version

This notebook tracks an object across an **uploaded video** (instead of a live webcam,
since Colab can't access your camera or open live windows).

**Steps:**
1. Install dependencies
2. Upload your video
3. Pick the object to track using sliders (on the first frame)
4. Run the tracker — it processes the whole video and saves an annotated output
5. Preview and download the result

Run the cells **in order**, from top to bottom.

## 1. Install dependencies

In [9]:
!pip install -q opencv-contrib-python ipywidgets
print("Done.")

Done.


## 2. Upload your video

In [10]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

Saving IMG_4660.MP4 to IMG_4660.MP4
Uploaded: IMG_4660.MP4


## 3. Choose the tracker algorithm\n\n- **CSRT** — most accurate, a bit slower (recommended)\n- **KCF** — faster, slightly less accurate\n- **MOSSE** — fastest, least accurate

In [11]:
TRACKER_NAME = "CSRT"  # change to "KCF" or "MOSSE" if you want

import cv2

TRACKERS = {
    "CSRT": cv2.TrackerCSRT_create,
    "KCF": cv2.TrackerKCF_create,
    "MOSSE": cv2.legacy.TrackerMOSSE_create,
}

def create_tracker(name):
    return TRACKERS[name]()

print(f"Using tracker: {TRACKER_NAME}")

Using tracker: CSRT


## 4. Select the object to track\n\nMove the sliders below until the green box tightly surrounds the object you want to track in the **first frame** of your video.

In [12]:
import cv2
from google.colab.patches import cv2_imshow
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(video_path)
ok, first_frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not read the video. Please check the uploaded file.")

frame_h, frame_w = first_frame.shape[:2]
print(f"Video resolution: {frame_w}x{frame_h}")

x_slider = widgets.IntSlider(min=0, max=frame_w - 1, step=1, value=frame_w // 4, description='x')
y_slider = widgets.IntSlider(min=0, max=frame_h - 1, step=1, value=frame_h // 4, description='y')
w_slider = widgets.IntSlider(min=1, max=frame_w, step=1, value=frame_w // 4, description='width')
h_slider = widgets.IntSlider(min=1, max=frame_h, step=1, value=frame_h // 4, description='height')

def show_box(x, y, bw, bh):
    preview = first_frame.copy()
    cv2.rectangle(preview, (x, y), (x + bw, y + bh), (0, 255, 0), 3)
    cv2_imshow(preview)

widgets.interact(show_box, x=x_slider, y=y_slider, bw=w_slider, bh=h_slider)

Video resolution: 464x848


interactive(children=(IntSlider(value=116, description='x', max=463), IntSlider(value=212, description='y', ma…

<function __main__.show_box(x, y, bw, bh)>

Once the green box looks right in the preview above, run this cell to lock in your selection:

In [17]:
bbox = (x_slider.value, y_slider.value, w_slider.value, h_slider.value)
print(f"Selected box (x, y, w, h): {bbox}")

Selected box (x, y, w, h): (116, 327, 162, 383)


## 5. Run the tracker on the whole video\n\nThis processes every frame, draws the tracked box, and saves the annotated video.

In [18]:
import time

cap = cv2.VideoCapture(video_path)
fps_in = cap.get(cv2.CAP_PROP_FPS) or 25
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ok, frame = cap.read()
if not ok:
    raise RuntimeError("Could not re-read the video.")

tracker = create_tracker(TRACKER_NAME)
tracker.init(frame, bbox)

output_raw = "tracked_output_raw.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_raw, fourcc, fps_in, (frame_w, frame_h))

frame_count = 0
lost_count = 0
start_time = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_count += 1

    tracked, box = tracker.update(frame)

    if tracked:
        x, y, w, h = [int(v) for v in box]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, "Tracking", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        lost_count += 1
        cv2.putText(frame, "Lost object!", (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.putText(frame, f"Tracker: {TRACKER_NAME}", (30, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    writer.write(frame)

    if frame_count % 30 == 0:
        print(f"Processed {frame_count}/{total_frames} frames...")

cap.release()
writer.release()

elapsed = time.time() - start_time
print(f"\nDone. Processed {frame_count} frames in {elapsed:.1f}s.")
if lost_count:
    print(f"Note: object was lost on {lost_count} frame(s).")

Processed 30/354 frames...
Processed 60/354 frames...
Processed 90/354 frames...
Processed 120/354 frames...
Processed 150/354 frames...
Processed 180/354 frames...
Processed 210/354 frames...
Processed 240/354 frames...
Processed 270/354 frames...
Processed 300/354 frames...
Processed 330/354 frames...

Done. Processed 353 frames in 59.8s.


## 6. Make the video browser-playable and preview it\n\nColab needs the video re-encoded with H.264 to preview it inline.

In [19]:
output_final = "tracked_output.mp4"
!ffmpeg -y -loglevel error -i {output_raw} -vcodec libx264 -pix_fmt yuv420p {output_final}

from IPython.display import HTML
from base64 import b64encode

video_data = open(output_final, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(video_data).decode()

HTML(f"""
<video width=600 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

## 7. Download the result

In [16]:
from google.colab import files
files.download(output_final)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes\n\n- If the object gets **lost** partway through, try re-running from step 4 and selecting a tighter box, or switch `TRACKER_NAME` to `\"KCF\"`.\n- Large/long videos take longer to process — a short clip (10–20 seconds) works best for testing.\n- This uses the same tracking logic as the local Python version (`object_tracking.py`), just adapted for Colab's upload/preview workflow.